# Cavanagh (1992): the paper's measures from one subject's runs

Reads every `*_trials.csv` under a data root (or one run folder) and computes what the
paper reports:

- **Equiluminance** (p. 1563, note 10): the k each colour stimulus was drawn with.
- **Motion null** (p. 1563): the luminance contrast at which the global rotation is nulled
  (5 to 10% in the paper).
- **Thresholds** (notes 6, 9; Fig. 1C): detection and direction thresholds for the luminance
  grating alone and with the 40% colour grating (raised "by a factor of 4 or 5"), and the
  colour detection threshold (40% is "about 20 times" it).
- **Tracking** (Fig. 1B): percentage of test pointers correctly identified against luminance
  contrast, colour-bar and luminance-bar sessions, and the 75% tracking threshold for
  luminance bars ("two to four times" the contrast at which luminance dominates the
  global motion).
- **Speed matching** (Fig. 2): the matched rate (Hz) of the comparison against the colour
  grating's luminance contrast, global and tracking matches; true speed 0.5 Hz.

Set `ANALYSIS_DATA` to a data root, or `ANALYSIS_RUN` to one run folder. Software
verification on simulated data shows that this analysis runs; it is not a replication.

In [ ]:
import os
import re
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt

where = Path(os.environ.get("ANALYSIS_RUN") or os.environ.get("ANALYSIS_DATA") or "../data")
files = sorted(where.rglob("*_trials.csv"))
frames = []
for f in files:
    m = re.search(r"_task-([a-z0-9-]+)", f.parent.name)
    df = pd.read_csv(f)
    df["task"] = m.group(1) if m else "unknown"
    df["run"] = f.parent.name
    frames.append(df)
trials = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["task", "completed"])
done = trials[trials["completed"].astype(bool)] if len(trials) else trials
print(f"{len(files)} runs, {len(trials)} trials, {len(done)} completed:")
print(done.groupby("task").size().to_string() if len(done) else "(none)")

In [ ]:
def gmean(x):
    x = np.asarray(x, dtype=float)
    return float(np.exp(np.log(x).mean()))


summary = {}
for task in ("equiluminance-speed", "flicker-photometry"):
    d = done[done.task == task]
    if len(d):
        summary[f"k ({task})"] = gmean(d.k_setting)
        print(f"{task}: k = {gmean(d.k_setting):.3f} from {len(d)} settings "
              f"(range {d.k_setting.min():.3f} to {d.k_setting.max():.3f})")

In [ ]:
null = done[done.task == "motion-null"]
if len(null):
    summary["motion null contrast"] = null.null_contrast.mean()
    print(null.groupby("start").null_contrast.mean().to_string())
    print(f"motion null: {null.null_contrast.mean():.1%} luminance contrast "
          f"(paper: 5 to 10%, depending on the observer)")

In [ ]:
thr = done[done.task == "thresholds"]
if len(thr):
    table = thr.groupby(["criterion", "grating", "mask"]).threshold.agg(["mean", "count"])
    print(table.to_string())
    for crit in thr.criterion.unique():
        cells = {key: table.loc[key, "mean"] for key in table.index if key[0] == crit}
        alone = cells.get((crit, "luminance", "none"), np.nan)
        masked = cells.get((crit, "luminance", "color"), np.nan)
        print(f"{crit}: masked / alone = {masked / alone:.1f} (paper: a factor of 4 or 5)")
        summary[f"{crit} threshold, luminance alone"] = alone
        summary[f"{crit} threshold, with colour"] = masked
    if ("presence", "color", "none") in table.index:
        c = table.loc[("presence", "color", "none"), "mean"]
        print(f"colour detection threshold {c:.2%}: 40% is {0.40 / c:.0f} times it (paper: about 20)")

In [ ]:
track = done[done.task == "attentive-tracking"].copy()
fig, ax = plt.subplots(figsize=(5, 3.4))
if len(track):
    track["correct"] = track.outcome == "CORRECT"
    curve = track.groupby(["target_grating", "luminance_contrast"]).correct.agg(["mean", "count"])
    print(curve.to_string())
    for target, marker in (("color", "o"), ("luminance", "s")):
        if target not in curve.index.get_level_values(0):
            continue
        c = curve.loc[target]
        ax.plot(c.index * 100, c["mean"] * 100, marker=marker, label=f"track {target} bars")
        # The first contrast reaching 75%, interpolated from the one below it.
        rising = c["mean"].to_numpy() >= 0.75
        xs, ys = c.index.to_numpy(), c["mean"].to_numpy()
        if rising.any():
            i = int(np.argmax(rising))
            t75 = xs[0] if i == 0 else xs[i - 1] + (0.75 - ys[i - 1]) * (xs[i] - xs[i - 1]) / (ys[i] - ys[i - 1])
            summary[f"{target} bars: 75% correct at"] = t75
            print(f"{target} bars: 75% correct at {t75:.1%} contrast" + (" (or below)" if i == 0 else ""))
            if target == "luminance" and "motion null contrast" in summary:
                print(f"  = {t75 / summary['motion null contrast']:.1f} x the motion null "
                      f"(paper: two to four times)")
ax.axhline(50, ls=":", color="grey")
ax.set_xlabel("Contrast of luminance grating (%)")
ax.set_ylabel("Tracking (% correct)")
ax.set_ylim(40, 102)
ax.legend(frameon=False)
ax.set_title("Fig. 1B measure")

In [ ]:
speed = done[done.task == "speed-match"]
fig, ax = plt.subplots(figsize=(5, 3.4))
if len(speed):
    table = speed.groupby(["judgment", "luminance_contrast"]).matched_rate_hz.agg(["mean", "sem", "count"])
    print(table.to_string())
    for judgment, marker in (("global", "o"), ("tracking", "s")):
        if judgment not in table.index.get_level_values(0):
            continue
        t = table.loc[judgment]
        ax.errorbar(t.index * 100, t["mean"], yerr=t["sem"].fillna(0), marker=marker,
                    label=f"{judgment} match", capsize=2)
        low = t["mean"].idxmin()
        summary[f"{judgment} match: slowest at"] = low
        print(f"{judgment} match: slowest ({t['mean'].min():.2f} Hz) at {low:+.0%} luminance contrast")
    ax.axhline(float(speed.outer_rate_hz.iloc[0]), ls=":", color="grey", label="true speed")
ax.set_xlabel("Luminance contrast of the colour grating (%)")
ax.set_ylabel("Speed match (Hz)")
ax.set_ylim(0, None)
ax.legend(frameon=False)
ax.set_title("Fig. 2 measure")

In [ ]:
pd.Series(summary, name="value").to_frame()